Load the CSV files and inspect their values

In [5]:
import pandas as pd
import numpy as np
import json

locations = pd.read_csv("../dataset/locations.csv")
incidents = pd.read_csv("../dataset/incidents.csv")
complaints = pd.read_csv("../dataset/public_complaints.csv")

print("Severity values:")
print(incidents["severity"].value_counts(dropna=False))

print("\nReported priority values:")
print(complaints["priority_reported"].value_counts(dropna=False))

print("\nIncident date range:")
print(incidents["timestamp"].min(), incidents["timestamp"].max())

print("\nComplaint date range:")
print(complaints["created_at"].min(), complaints["created_at"].max())

Severity values:
severity
Low         3819
Medium      3726
High        1985
Critical     470
Name: count, dtype: int64

Reported priority values:
priority_reported
Normal    4469
Low       1943
High      1588
Name: count, dtype: int64

Incident date range:
2025-01-01 00:29:00 2025-12-31 23:50:00

Complaint date range:
2025-01-01 00:03:00 2025-12-31 21:31:00


Process the data and calculate the divergence

In [6]:
# 1. Clean IDs and dates
for df in [locations, incidents, complaints]:
    df["location_id"] = df["location_id"].astype(str).str.strip()

incidents["timestamp"] = pd.to_datetime(
    incidents["timestamp"], errors="coerce"
)
complaints["created_at"] = pd.to_datetime(
    complaints["created_at"], errors="coerce"
)

# 2. Define weights.
# Customize these mappings to match the values printed in Cell 1.
severity_weights = {
    "Low": 1,
    "Medium": 2,
    "High": 3,
    "Critical": 4,
}

priority_weights = {
    "Low": 1,
    "Normal": 2,
    "High": 3,
}

def apply_weights(series, mapping):
    values = series.astype("string").str.strip().str.lower()
    # Unknown or missing labels receive a neutral default weight of 1.
    return values.map(mapping).fillna(1).astype(float)

incidents["incident_weight"] = apply_weights(
    incidents["severity"], severity_weights
)

complaints["complaint_weight"] = apply_weights(
    complaints["priority_reported"], priority_weights
)

# 3. Aggregate incident signals by location
incident_summary = (
    incidents.groupby("location_id")
    .agg(
        incident_count=("location_id", "size"),
        weighted_incidents=("incident_weight", "sum"),
        injury_count=("injuries", "sum"),
        last_incident=("timestamp", "max"),
    )
    .reset_index()
)

# 4. Aggregate complaint signals by location
complaint_summary = (
    complaints.groupby("location_id")
    .agg(
        complaint_count=("location_id", "size"),
        weighted_complaints=("complaint_weight", "sum"),
        avg_days_to_resolution=("days_to_resolution", "mean"),
        last_complaint=("created_at", "max"),
    )
    .reset_index()
)

# 5. Start with ALL locations, retaining zero-event locations
features = (
    locations
    .merge(incident_summary, on="location_id", how="left")
    .merge(complaint_summary, on="location_id", how="left")
)

count_columns = [
    "incident_count",
    "weighted_incidents",
    "injury_count",
    "complaint_count",
    "weighted_complaints",
]

features[count_columns] = features[count_columns].fillna(0)

# 6. Calculate z-scores across locations
def z_score(series):
    std = series.std(ddof=0)
    if pd.isna(std) or std == 0:
        return pd.Series(0.0, index=series.index)
    return (series - series.mean()) / std

features["z_incidents"] = z_score(
    features["weighted_incidents"]
)

features["z_complaints"] = z_score(
    features["weighted_complaints"]
)

# 7. Divergence = complaint z-score - incident z-score
features["divergence"] = (
    features["z_complaints"] - features["z_incidents"]
)

# 8. Classify and explain each location
def classify_divergence(row):
    if row["incident_count"] == 0 and row["complaint_count"] == 0:
        return "insufficient_data"

    d = row["divergence"]

    if d >= 1.0:
        return "potential_blind_spot"
    elif d <= -1.0:
        return "potential_reporting_gap"
    else:
        return "signals_aligned"

def explain_divergence(row):
    category = row["divergence_category"]

    if category == "potential_blind_spot":
        return (
            "Complaint signal is higher than the official incident "
            "signal relative to other locations. Investigate a possible "
            "recording or reporting gap."
        )
    elif category == "potential_reporting_gap":
        return (
            "Official incident signal is higher than the complaint "
            "signal relative to other locations. Investigate a possible "
            "public reporting gap."
        )
    elif category == "insufficient_data":
        return "No incidents or complaints were recorded in this dataset."
    else:
        return "Complaint and incident signals are relatively aligned."

features["divergence_category"] = features.apply(
    classify_divergence, axis=1
)
features["explanation"] = features.apply(
    explain_divergence, axis=1
)

# 9. Sort the largest divergences first
features["abs_divergence"] = features["divergence"].abs()

features = features.sort_values(
    "abs_divergence", ascending=False
).reset_index(drop=True)

features["rank"] = np.arange(1, len(features) + 1)

# Inspect results
display(
    features[
        [
            "location_id",
            "zone",
            "latitude",
            "longitude",
            "incident_count",
            "complaint_count",
            "z_incidents",
            "z_complaints",
            "divergence",
            "divergence_category",
        ]
    ].head(20)
)

,location_id,zone,latitude,longitude,incident_count,complaint_count,z_incidents,z_complaints,divergence,divergence_category
0,L0683,ZONE_06,16.486752,80.548342,3,14,-2.153082,2.096058,4.249140,potential_blind_spot
1,L0679,ZONE_11,16.343721,80.616552,6,16,-1.230333,2.794744,4.025076,potential_blind_spot
2,L0136,ZONE_08,16.437282,80.429180,18,4,2.460665,-1.397372,-3.858037,potential_reporting_gap
3,L0358,ZONE_07,16.420528,80.478339,19,5,2.768248,-1.048029,-3.816277,potential_reporting_gap
4,L0521,ZONE_15,16.622104,80.407137,21,7,3.383414,-0.349343,-3.732757,potential_reporting_gap
5,L0646,ZONE_01,16.502526,80.484818,7,16,-0.922749,2.794744,3.717493,potential_blind_spot
6,L0362,ZONE_05,16.440655,80.469736,23,9,3.998581,0.349343,-3.649238,potential_reporting_gap
7,L0702,ZONE_07,16.410185,80.541087,4,13,-1.845499,1.746715,3.592214,potential_blind_spot
8,L0709,ZONE_08,16.472817,80.512367,4,13,-1.845499,1.746715,3.592214,potential_blind_spot
9,L0967,ZONE_05,16.421535,80.612137,17,4,2.153082,-1.397372,-3.550454,potential_reporting_gap


Export the output

In [7]:
# Fields that the frontend needs
output_columns = [
    "location_id",
    "zone",
    "latitude",
    "longitude",
    "road_type",
    "road_condition",
    "near_school",
    "near_hospital",
    "pedestrian_activity",
    "incident_count",
    "weighted_incidents",
    "injury_count",
    "complaint_count",
    "weighted_complaints",
    "avg_days_to_resolution",
    "z_incidents",
    "z_complaints",
    "divergence",
    "divergence_category",
    "explanation",
    "rank",
]

map_output = features[output_columns].copy()

# Convert non-finite numbers and missing values to JSON null
map_output = map_output.replace([np.inf, -np.inf], np.nan)

map_output.to_json(
    "module3_divergence.json",
    orient="records",
    indent=2
)

print(f"Exported {len(map_output)} locations")
print("Saved: module3_divergence.json")
print(map_output.head(3).to_json(orient="records", indent=2))

Exported 1000 locations
Saved: module3_divergence.json
[
  {
    "location_id":"L0683",
    "zone":"ZONE_06",
    "latitude":16.486752,
    "longitude":80.548342,
    "road_type":"Collector",
    "road_condition":"Fair",
    "near_school":1,
    "near_hospital":1,
    "pedestrian_activity":"Medium",
    "incident_count":3,
    "weighted_incidents":3.0,
    "injury_count":0,
    "complaint_count":14,
    "weighted_complaints":14.0,
    "avg_days_to_resolution":5.7857142857,
    "z_incidents":-2.1530818817,
    "z_complaints":2.0960578781,
    "divergence":4.2491397598,
    "divergence_category":"potential_blind_spot",
    "explanation":"Complaint signal is higher than the official incident signal relative to other locations. Investigate a possible recording or reporting gap.",
    "rank":1
  },
  {
    "location_id":"L0679",
    "zone":"ZONE_11",
    "latitude":16.343721,
    "longitude":80.616552,
    "road_type":"Local",
    "road_condition":"Fair",
    "near_school":0,
    "near_hosp